In this notebook, we will investigate the ART matrix and see if we can find a pressure domain PU matrix that matches the ART energy matrix, $\mathbf{S}$ exactly. The ART matrix is column-stochastic, i.e, 
$$
\begin{aligned}
\sum_h S_{h\to i \to j} &= 1 \quad \forall\, i,j \quad \text{(lossless)}, \\
\mathbf{1}^\top \mathbf{S} &= \mathbf{1}^\top.
\end{aligned}
$$

Now, we want to find a PU matrix, $\mathbf{H}(e^{j\omega})$ such that
$$\frac{1}{2 \pi} \int_0^{2\pi}|H_{ij}(e^{j\omega})|^2 d\omega = S_{ij}, \quad \mathbf{H}(e^{j\omega})\mathbf{H}(e^{-j\omega})^\top = \mathbf{I}$$

Note that $|\mathbf{H}(e^{j\omega})|^{\odot 2}$ is doubly stochastic. However, $\mathbf{S}$ is only column-stochastic. The best we can do is parameterize $\mathbf{H}(z) = \mathbf{U}_0 \Pi_{k=1}^K (\mathbf{I} - (1-z^{-1})v_k v_k^\top) = \sum_{k=0}^K \mathbf{U}_k z^{-k}$. This gives us, 
$$\frac{1}{2 \pi} \int_0^{2\pi}|\mathbf{H}(e^{j\omega})|^{\odot 2} d\omega = \sum_k |\mathbf{U}_k|^{\odot 2}$$
So we end up solving the (highly non-convex and non-linear) optimisation problem which will always have a non-zero error:

$$\arg \min_{v_1, \ldots, v_K} ||\sum_k |\mathbf{U}_k|^{\odot 2} - \mathbf{S}||_F^2 \quad \text{s.t. } v_k^\top v_k =1 \ \forall k$$

Why do we insist on finding a PU matrix, instead of simply solving $\arg \min_{\mathbf{U}} || |\mathbf{U}|^{\odot 2} - \mathbf{S}||_F^2 \quad \text{s.t. } \mathbf{U}^\top \mathbf{U} = \mathbf{I}$? For a pressure vector, $p \in \mathbb{R}^N$ and its corresponding energy vector $e \in \mathbb{R}^N$ where $e = |p|^{\odot 2}$, we want something like
$$
\begin{aligned}
\mathbf{S}e &= |\mathbf{U}p|^{\odot 2} \\
\mathbf{S}|p|^{\odot 2} &= |\mathbf{U}p|^{\odot 2} \\
\text{BUT, } |\mathbf{U}|^{\odot 2} |p|^{\odot 2} &= |\mathbf{U}p|^{\odot 2} + \text{cross terms}
\end{aligned}
$$

The cross terms don't vanish automatically. However, by using a paraunitary operator on $p$, we decorrelate the different channels of $p$ and this makes the cross-terms disappear. So if we could exactly solve the optimisation problem such that $\sum_k |\mathbf{U}_k|^{\odot 2} = \mathbf{S}$, then 
$$
\begin{aligned}
\mathbf{H}(z) p &= \sum_k \mathbf{U}_k p z^{-k} \\
\int |\mathbf{H}(e^{j\omega}) p|^{\odot 2} &= \int |\sum_k \mathbf{U}_k p e^{-j\omega_k}|^{\odot 2} d\omega \\
&= \int \left( \sum_k \mathbf{U}_k p e^{-j\omega_k} \right) \odot \left(\sum_m \mathbf{U}_m^* p^* e^{-j\omega_m}\right) d\omega \\
&=  \left( \int \sum_k \sum_m \mathbf{U}_k \odot \mathbf{U}_m^* e^{-j(\omega_k - \omega_m)} d\omega\right) \ |p|^{\odot 2} \\
&= \left( \sum_k \mathbf{U}_k \odot \mathbf{U}_k^*\right) |p|^{\odot 2} \\
&= |\sum_k \mathbf{U}_k|^{\odot 2} |p|^{\odot 2} = \mathbf{S} |p|^{\odot 2}
\end{aligned}
$$

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import scipy
import os
from pathlib import Path
from loguru import logger

from slope2noise.utils import db
import SDNPy.utils.ARTMatrixMath as mm
import SDNPy.utils.visualize_room as vis
from SDNPy.DataTypes import DelayConstructionType

from raves.src.utils.raves_io import load_all_inputs

In [ ]:
environment_name = 'ERTD_1_patch_per_wall'
environment_folder = os.path.join('..', 'environment', environment_name)
sample_rate = 44100

fig_path = Path('../../../Figures/ART/ERTD/matrices')
fig_path.mkdir(parents=True, exist_ok=True)

#### Set up logging

In [ ]:
logs_dir = Path(f"{fig_path}/{environment_name}/logs")
logs_dir.mkdir(parents=True, exist_ok=True)
log_path = logs_dir / f"ertd_column_stochastic_to_paraunitary_logger.txt"
if log_path.exists():
    log_path.unlink()
logger.add(log_path, enqueue=True, backtrace=False, diagnose=False)

#### Read ART patching matrix and reflection matrix

In [ ]:
# Read the .mtx file
art_reflect_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_band_1.mtx')
art_diffuse_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_diffuse.mtx')
art_specular_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_specular.mtx')
art_patching_matrix = scipy.io.mmread(f'{environment_folder}/path_indexing.mtx')

# Convert to dense format if needed for visualization (optional)
art_full_dense_matrix = art_reflect_matrix.toarray()
art_diffuse_dense_matrix = art_diffuse_matrix.toarray()
art_specular_dense_matrix = art_specular_matrix.toarray()
art_patching_matrix_dense = art_patching_matrix.toarray()

num_non_zero_elems = np.count_nonzero(art_patching_matrix_dense)
assert art_full_dense_matrix.shape[0] == num_non_zero_elems

# Extract patch labels in mesh order from mesh.obj
mesh_obj_path = Path(environment_folder) / 'mesh.obj'
patch_labels = []
with open(mesh_obj_path, 'r', encoding='utf-8') as f:
    for line in f:
        line_no_comment = line.split('#', 1)[0].strip()
        if not line_no_comment.startswith('usemtl '):
            continue
        mat_name = line_no_comment.split()[1]
        if mat_name not in patch_labels:
            patch_labels.append(mat_name)

num_patches = art_patching_matrix_dense.shape[0]
assert len(patch_labels) == num_patches, (
    f'Expected {num_patches} patch labels from mesh.obj, found {len(patch_labels)}.'
)

#### Get the sparse ART diffuse and specular matrices' pruned, dense version for each patch

In [ ]:
label = ['diffuse', 'specular']
for k, art_dense_matrix in enumerate([art_diffuse_dense_matrix, art_specular_dense_matrix]):
    
    art_matrix_pruned_list = mm.prune_tdart_matrix(art_dense_matrix, art_patching_matrix.copy())
    if k == 0:
        art_diffuse_matrix_pruned_list = art_matrix_pruned_list.copy()
    elif k == 1:
        art_specular_matrix_pruned_list = art_matrix_pruned_list.copy()
        

#### Read mesh and patch properties to create proper ART reflection matrix without absorption

In [ ]:
mesh, patch_materials, material_coefficients, environment_folder = load_all_inputs(environment_folder, area_threshold=0, thoroughness=0)
assert len(patch_materials) == num_patches, "Number of patch materials must match number of patches"
patch_absorption = []
patch_scattering = []
art_matrix_pruned_list = []

for band_idx, center_frequency in enumerate(material_coefficients['Frequencies']):

    for i, patch_mat in enumerate(patch_materials):
        # Retrieve the coefficients of patch i for this frequency band.
        patch_absorption.append(material_coefficients[patch_mat][0, band_idx])
        patch_scattering.append(material_coefficients[patch_mat][1, band_idx])

        art_matrix_pruned_list.append(art_diffuse_matrix_pruned_list[i] * patch_scattering[i] + 
                                      art_specular_matrix_pruned_list[i] * (1 - patch_scattering[i]))


fig, axes = plt.subplots(4, 4, figsize=(10, 10), constrained_layout=True)
for i, ax in enumerate(axes.flat):
    # make sure no zero elements in the matrices
    num_elems =  art_matrix_pruned_list[i].shape[0] * art_matrix_pruned_list[i].shape[1]
    assert np.count_nonzero(art_matrix_pruned_list[i]) == num_elems
    ax.imshow(art_matrix_pruned_list[i], cmap='viridis', interpolation='nearest')
    ax.set_title(f"{patch_labels[i]}", fontsize=8)
    ax.set_xlabel("i -> j")
    ax.set_ylabel("h -> i")

plt.savefig(f'{fig_path.resolve()}/{environment_name}_stochastic_art_matrices.png')    
plt.show()

#### Find the FIR PU matrix whose average power gives the desired column stochastic matrix

#### The fixed unitary matrix is from SA optimisation

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10), constrained_layout=True)

art_matrix_pu_list = []
for i, ax1 in enumerate(axes.flat):
    try:
        cur_art_matrix = art_matrix_pruned_list[i].copy()
        cur_mat_num_rows = cur_art_matrix.shape[0]
        start_time = time.perf_counter()
        cur_art_pu_matrix, info = mm.single_stochastic_to_paraunitary(cur_art_matrix, 
                                                                      num_stages=32,
                                                                      return_info=True, 
                                                                      use_sa_unitary_init=True,)
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        
        art_matrix_pu_list.append(cur_art_pu_matrix) 
        ax1.imshow(info["average_energy"], cmap='viridis', interpolation='nearest')
        ax1.set_title(f"{patch_labels[i]}", fontsize=8)
        ax1.set_xlabel("i -> j")
        ax1.set_ylabel("h -> i")
    
        logger.info(f"PU reconstruction error: {db(info['average_energy_error']):.4f} dB")
        logger.info(f"Time taken: {execution_time:.3f}s")
        assert db(info['average_energy_error']) <= -6, "PU reconstriction failed"
        assert info["is_paraunitary"], "Resulting matrix not unitary"
    except AssertionError as e:
        logger.error(f"PU reconstruction failed for {patch_labels[i]} scattering matrix of size {cur_mat_num_rows} x {cur_mat_num_rows}")
        continue
    
           
    # plot polynomial matrix
    vis.plot_poly_mats_overlay([cur_art_pu_matrix], 
                               labels=[patch_labels[i]], 
                               fs=sample_rate, 
                               title=f"{patch_labels[i]}, PU recons err ={db(info['average_energy_error']):.4f}dB",
                               save_path = f'{fig_path.resolve()}/{environment_name}_pu_art_matrix_{patch_labels[i]}_single_stochastic_sa_init.png',
                              )


fig.savefig(f'{fig_path.resolve()}/{environment_name}_pu_single_stochastic_sa_init_avg_energy_art_matrices.png')
plt.show()

band_idx = 0
center_frequency = 0
save_folder = Path(f'{environment_folder}_pressure_domain_pu_from_ss_reflection_matrix/')
save_folder.mkdir(parents=True, exist_ok=True)
save_path = save_folder / f'ART_kernel_band_{band_idx+1}'
art_pu_matrix_full = mm.reconstruct_full_tdart_matrix(art_matrix_pu_list, art_patching_matrix_dense, 
                                                      patch_absorption=patch_absorption, save_path=save_path)

#### No initialisation with SA optimisation

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10), constrained_layout=True)

art_matrix_pu_list = []
for i, ax1 in enumerate(axes.flat):
    try:
        cur_art_matrix = art_matrix_pruned_list[i].copy()
        cur_mat_num_rows = cur_art_matrix.shape[0]
        start_time = time.perf_counter()
        cur_art_pu_matrix, info = mm.single_stochastic_to_paraunitary(cur_art_matrix, 
                                                                      num_stages=32,
                                                                      return_info=True, 
                                                                      use_sa_unitary_init=False,)
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        
        art_matrix_pu_list.append(cur_art_pu_matrix) 
        ax1.imshow(info["average_energy"], cmap='viridis', interpolation='nearest')
        ax1.set_title(f"{patch_labels[i]}", fontsize=8)
        ax1.set_xlabel("i -> j")
        ax1.set_ylabel("h -> i")
    
        logger.info(f"PU reconstruction error: {db(info['average_energy_error']):.4f} dB")
        logger.info(f"Time taken: {execution_time:.3f}s")
        assert db(info['average_energy_error']) <= -6, "PU reconstriction failed"
        assert info["is_paraunitary"], "Resulting matrix not unitary"
    except AssertionError as e:
        logger.error(f"PU reconstruction failed for {patch_labels[i]} scattering matrix of size {cur_mat_num_rows} x {cur_mat_num_rows}")
        continue
    
           
    # plot polynomial matrix
    vis.plot_poly_mats_overlay([cur_art_pu_matrix], 
                               labels=[patch_labels[i]], 
                               fs=sample_rate, 
                               title=f"{patch_labels[i]}, PU recons err ={db(info['average_energy_error']):.4f}dB",
                               save_path = f'{fig_path.resolve()}/{environment_name}_pu_art_matrix_{patch_labels[i]}_single_stochastic.png',
                              )


fig.savefig(f'{fig_path.resolve()}/{environment_name}_pu_single_stochastic_avg_energy_art_matrices.png')
plt.show()